In [1]:
import pandas as pd
from scipy.signal import argrelextrema
import numpy as np
from pylab import *
import mpmath as mp
from lmfit import minimize, Minimizer, Parameters, Parameter, report_fit
from IPython.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))
import matplotlib as mpl
import matplotlib.pyplot as plt
%matplotlib tk

data = pd.read_csv(r'https://raw.githubusercontent.com/jerrengrimes/Projects/refs/heads/main/EIS%20Interactive/example_eis_data.csv')

In [ ]:
from matplotlib.widgets import Slider, Button

ff = np.logspace(-6,8,1000); 
w = 2*np.pi*ff;

Lind = 2.68E-07
Rel = -0.017360688
L = 1.98E-05
Ri = 8.82667984
R1 = 2.876E-06
f1 = 2304.419638
n1 = 0.7463828
RW = 0.02682548047319
LL = 0.000340084
D = 4.5491E-06
nw = 0.470403253 # 0< nw < 0.5
Ind = Lind * 1j*w
               
Q1 = (1 / (R1 * (2 * np.pi * f1) ** n1));
Phi = (R1 / (1 + R1 * Q1 * (w * 1j) ** n1));
X1 = Ri
Lam = (Phi / X1) ** (1 / 2)

x = L / Lam
x_mp = mp.matrix(x)  # x in mp.math format
coth_mp = []
for i in range(len(Lam)):
    coth_mp.append(float(mp.coth(x_mp[i]).real) + float(mp.coth(x_mp[i]).imag) * 1j)  # Handles coth with x having very large or very small numbers

Z_TLs = Lam * X1 * coth_mp

#L = effective diffusion thickness (m);    
#D = effective diffusion coefficient of the particle (m);

T = LL**2/D;
Z_W = (RW*tanh((1j*T*w)**(nw)))/(1j*T*w)**(nw)
Ztotal = Ind + Rel + Z_TLs + Z_W;

# Create the figure and the line that we will manipulate
fig, ax = plt.subplots(nrows = 1, ncols = 2, figsize=(14,10))

ax1 = ax[0]
line3, = ax1.plot(Ztotal.real, -Ztotal.imag, lw=2)
line4, = ax1.plot(data.iloc[:,1].dropna(),data.iloc[:,2].dropna(),linestyle='none',color='black',label='Data',marker = 'o',fillstyle = 'none')
ax1.set_xlim(0.5*min(data.iloc[:,1]), 1.05*max(data.iloc[:,1]));
ax1.set_ylim(-0.02, 1.5*max(data.iloc[:,2]))
ax1.set_xlabel('$Z_{real}$')
ax1.set_ylabel('$-Z_{imag}$')
ax1.grid()

ax2 = ax[1]
line5, = ax2.plot(data.iloc[:,0].dropna(),data.iloc[:,2].dropna(),linestyle='none',color='black',label='Data',marker = 'o',fillstyle = 'none')
line6, = ax2.plot(ff, -Ztotal.imag, lw=2)
ax2.set_xlim(0.01, 10**6)
ax2.set_ylim(-0.02, 1.5*max(data.iloc[:,2]))
ax2.set_xscale('log');
ax2.set_xlabel('$Z_{real}$')
ax2.set_ylabel('$-Z_{imag}$')
ax2.grid()

# adjust the main plot to make room for the sliders
plt.subplots_adjust(left=0.25, bottom=0.55, top=0.95)

axLind = plt.axes([0.25, 0.47, 0.65, 0.03])
Lind_slider = Slider(ax=axLind,label='$L_{inductance}$',valmin=0.1*Lind,valmax=5*Lind,valinit=Lind,)

axrel = plt.axes([0.25, 0.44, 0.65, 0.03])
rel_slider = Slider(ax=axrel,label='$R_{electrolyte}$',valmin=1.5*Rel,valmax=-5*Rel,valinit=Rel,)

axL = plt.axes([0.25, 0.41, 0.65, 0.03])
L_slider = Slider(ax=axL,label='Effective Diffusion Thickness',valmin=0.1*L,valmax=5*L,valinit=L,)

axRi = plt.axes([0.25, 0.38, 0.65, 0.03])
Ri_slider = Slider(ax=axRi,label='YSZ Ionic Resistance',valmin=0.00001,valmax=1000,valinit=Ri,)
#Ri_slider = Slider(ax=axRi,label='Ri',valmin=0.1*Ri,valmax=5*Ri,valinit=Ri,)
axR1 = plt.axes([0.25, 0.35, 0.65, 0.03])
R1_slider = Slider(ax=axR1,label='R1',valmin=0.1*R1,valmax=5*R1,valinit=R1,)

axf1 = plt.axes([0.25, 0.32, 0.65, 0.03])
f1_slider = Slider(ax=axf1,label='Peak Frequency 1',valmin=0.1*f1,valmax=5*f1,valinit=f1,)

axn1 = plt.axes([0.25, 0.29, 0.65, 0.03])
n1_slider = Slider(ax=axn1,label='n1',valmin=0,valmax=1,valinit=n1,)

axRW = plt.axes([0.25, 0.26, 0.65, 0.03])
RW_slider = Slider(ax=axRW,label='$R_{Warburg}$',valmin=0.1*RW,valmax=5*RW,valinit=RW,)

axLL = plt.axes([0.25, 0.23, 0.65, 0.03])
LL_slider = Slider(ax=axLL,label='Electrode Thickness',valmin=0.1*LL,valmax=5*LL,valinit=LL,)

axD = plt.axes([0.25, 0.20, 0.65, 0.03])
D_slider = Slider(ax=axD,label='Electrode Diffusivity',valmin=0.1*D,valmax=5*D,valinit=D,)

axnw = plt.axes([0.25, 0.17, 0.65, 0.03])
nw_slider = Slider(ax=axnw,label='$n_{Warburg}$',valmin=0,valmax=0.5,valinit=nw,)

# The function to be called anytime a slider's value changes
def update(val):

    rel = rel_slider.val; 
    Lind = Lind_slider.val; 
    
    L = L_slider.val; 
    Ri = Ri_slider.val; 
    R1 = R1_slider.val; 
    f1 = f1_slider.val; 
    n1 = n1_slider.val
    
    
    RW = RW_slider.val
    LL = LL_slider.val; 
    D = D_slider.val;
    nw = nw_slider.val;
    
    Rel_arr = [rel]*1000;
    Ind = Lind * 1j*w;

    Q1 = (1 / (R1 * (2 * np.pi * f1) ** n1));
    Phi = (R1 / (1 + R1 * Q1 * (w * 1j) ** n1));
    X1 = Ri
    Lam = (Phi / X1) ** (1 / 2)

    x = L / Lam
    x_mp = mp.matrix(x)  # x in mp.math format
    coth_mp = []
    for i in range(len(Lam)):
        coth_mp.append(float(mp.coth(x_mp[i]).real) + float(
            mp.coth(x_mp[i]).imag) * 1j)  # Handles coth with x having very large or very small numbers

    Z_TLs = Lam * X1 * coth_mp
    T = (LL**2)/D;
    Z_W = (RW*tanh((1j*T*w)**(nw)))/((1j*T*w)**(nw));

    Ztotal = Ind + Rel_arr + Z_TLs + Z_W;

    line3.set_data(Ztotal.real,-Ztotal.imag)
    line6.set_data(ff, -Ztotal.imag)
    fig.canvas.draw_idle()


# register the update function with each slider

Lind_slider.on_changed(update)
rel_slider.on_changed(update)
L_slider.on_changed(update)
R1_slider.on_changed(update)
f1_slider.on_changed(update)
n1_slider.on_changed(update)
Ri_slider.on_changed(update)
RW_slider.on_changed(update)
LL_slider.on_changed(update)
D_slider.on_changed(update)
nw_slider.on_changed(update)

plt.show()